# 14 Gold Encounter Utilization Summary

## Purpose

This notebook creates an enterprise healthcare encounter utilization analytics table.

## What We Are Doing

We will analyze patient encounter patterns including:
- ambulatory visits
- emergency visits
- inpatient admissions
- encounter durations
- utilization burden
- operational healthcare metrics

## Why We Are Doing This

Encounter utilization is one of the most important healthcare analytics domains.

This table supports:
- hospital operations analytics
- utilization management
- payer/provider analytics
- capacity planning
- population health
- ML risk scoring

## Final Output

`healthcare_catalog.gold.encounter_utilization_summary`

## Final Grain

One row per patient.

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
We need aggregation, timestamp, and KPI calculation functions.

### Expected Output
Spark functions available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Encounter Silver Table

### What We Are Doing
We are reading the clean encounter table from the Silver layer.

### Why We Are Doing This
Gold utilization analytics should be built from clean normalized Silver data.

### Expected Output
Encounter DataFrame loaded successfully.

In [0]:
encounter_df = spark.table(
    "healthcare_catalog.silver.encounter_clean"
)

print("Encounter Silver table loaded successfully.")

Encounter Silver table loaded successfully.


## Step 3 — Inspect Encounter Data

### What We Are Doing
We are reviewing encounter data.

### Why We Are Doing This
We want to verify:
- encounter classes
- timestamps
- duration fields
- patient linkage

### Expected Output
Encounter-level healthcare utilization data.

In [0]:
display(encounter_df)

encounter_id,patient_reference,encounter_status,encounter_class,encounter_start,encounter_end,service_provider,encounter_type,length_of_stay_hours,patient_id,encounter_type_text
a889805b-20ca-0ba6-5b5c-e617ce99b0cc,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1954-04-03T17:14:47.000Z,1954-04-03T17:29:47.000Z,MOUNT AUBURN HOSPITAL,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""185347001"",""display"":""Encounter for problem""}],""text"":""Encounter for problem""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Encounter for problem
b1ca5d4d-e3b5-e65b-e900-3844ae30f421,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1954-04-20T13:14:47.000Z,1954-04-20T13:29:47.000Z,MOUNT AUBURN HOSPITAL,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""185347001"",""display"":""Encounter for problem""}],""text"":""Encounter for problem""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Encounter for problem
853a4e65-da0d-d531-eb6e-458a70bcf552,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1959-06-03T19:14:47.000Z,1959-06-03T19:33:08.000Z,MOUNT AUBURN HOSPITAL,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""185345009"",""display"":""Encounter for symptom""}],""text"":""Encounter for symptom""}]",0.30583333333333335,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Encounter for symptom
b04db379-2092-c969-8968-39ab3557b605,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1971-01-29T08:14:47.000Z,1971-01-29T08:29:47.000Z,PCP58532,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""162673000"",""display"":""General examination of patient (procedure)""}],""text"":""General examination of patient (procedure)""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,General examination of patient (procedure)
dffdcaf9-384a-3472-867c-2f852dc61851,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1977-10-25T08:14:47.000Z,1977-10-25T08:29:47.000Z,MOUNT AUBURN HOSPITAL,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""185345009"",""display"":""Encounter for symptom""}],""text"":""Encounter for symptom""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Encounter for symptom
ed03c4f1-9865-7638-37d1-d7c1fe282363,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1978-02-10T08:14:47.000Z,1978-02-10T08:29:47.000Z,PCP58532,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""162673000"",""display"":""General examination of patient (procedure)""}],""text"":""General examination of patient (procedure)""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,General examination of patient (procedure)
e756ebcd-40d3-59f6-257c-4b413d4e94e8,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1981-02-13T08:14:47.000Z,1981-02-13T08:29:47.000Z,PCP58532,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""162673000"",""display"":""General examination of patient (procedure)""}],""text"":""General examination of patient (procedure)""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,General examination of patient (procedure)
e47502eb-e574-2058-f1f9-4f19844e4da8,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1982-04-22T22:14:47.000Z,1982-04-22T22:37:23.000Z,MOUNT AUBURN HOSPITAL,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""185345009"",""display"":""Encounter for symptom""}],""text"":""Encounter for symptom""}]",0.37666666666666665,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Encounter for symptom
e44915f6-9666-521e-7c74-6f5a5035db88,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1984-02-17T08:14:47.000Z,1984-02-17T08:29:47.000Z,PCP58532,"[{""coding"":[{""system"":""http://snomed.info/sct"",""code"":""162673000"",""display"":""General examination of patient (procedure)""}],""text"":""General examination of patient (procedure)""}]",0.25,b0a06ead-cc42-aa48-dad6-841d4aa679fa,General examination of patient (procedure)
1fff2e1c-2dc4-c32d-4894-c1a62c7c80be,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,1987-02-20T08

## Step 4 — Create Patient-Level Encounter Utilization Metrics

### What We Are Doing
We are aggregating healthcare utilization metrics by patient.

### Why We Are Doing This
Patient-level utilization is critical for:
- operational analytics
- risk stratification
- healthcare cost analysis
- ML feature engineering

### Metrics Created

- total encounters
- ambulatory encounters
- emergency encounters
- inpatient encounters
- average encounter duration
- maximum encounter duration
- total encounter hours

### Expected Output
One row per patient with utilization KPIs.

In [0]:
encounter_with_duration_df = encounter_df.withColumn(
    "encounter_duration_hours",
    (
        unix_timestamp(col("encounter_end")) -
        unix_timestamp(col("encounter_start"))
    ) / 3600
)

encounter_utilization_df = encounter_with_duration_df.groupBy(
    "patient_id"
).agg(

    count("*").alias("total_encounters"),

    sum(
        when(col("encounter_class") == "AMB", 1).otherwise(0)
    ).alias("ambulatory_encounters"),

    sum(
        when(col("encounter_class") == "EMER", 1).otherwise(0)
    ).alias("emergency_encounters"),

    sum(
        when(col("encounter_class") == "IMP", 1).otherwise(0)
    ).alias("inpatient_encounters"),

    avg("encounter_duration_hours").alias("avg_encounter_duration_hours"),

    max("encounter_duration_hours").alias("max_encounter_duration_hours"),

    sum("encounter_duration_hours").alias("total_encounter_hours")
)

display(encounter_utilization_df)

patient_id,total_encounters,ambulatory_encounters,emergency_encounters,inpatient_encounters,avg_encounter_duration_hours,max_encounter_duration_hours,total_encounter_hours
b0a06ead-cc42-aa48-dad6-841d4aa679fa,32,30,2,0,0.30257812500000003,1.0,9.682500000000001
ccfc4db2-2026-7adb-3db0-33f3828140bb,38,37,1,0,0.26973684210526316,1.0,10.25
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,64,63,0,1,0.62109375,24.0,39.75
71a8b156-760b-df6b-859e-eefc7932a526,19,18,1,0,0.2894736842105263,1.0,5.5
76b289fd-e825-734c-8446-316f59643593,24,21,2,1,9.409189814814814,216.56722222222223,225.82055555555556
92fb7efc-5cfd-f8d3-927b-42f8ee099531,27,25,2,0,0.35032921810699585,1.0947222222222222,9.458888888888888
81aa7647-779f-fd6b-94cf-782e606efeb2,17,17,0,0,0.25,0.25,4.25
346a1435-2455-914f-c287-7b88052d05db,69,65,4,0,0.3047342995169082,1.0,21.026666666666667
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,38,37,1,0,5.659824561403509,205.0,215.07333333333335
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,54,48,6,0,123.94552469135805,6672.0,6693.058333333334


## Step 5 — Create Utilization Risk Flags

### What We Are Doing
We are creating operational healthcare risk indicators.

### Why We Are Doing This
Healthcare systems use utilization thresholds for:
- high-risk patient detection
- operational monitoring
- payer analytics
- population health

### Expected Output
Risk flag columns added.

In [0]:
encounter_utilization_df = encounter_utilization_df.withColumn(

    "high_emergency_utilization_flag",

    when(col("emergency_encounters") >= 3, 1).otherwise(0)

).withColumn(

    "high_inpatient_utilization_flag",

    when(col("inpatient_encounters") >= 2, 1).otherwise(0)

).withColumn(

    "high_total_utilization_flag",

    when(col("total_encounters") >= 50, 1).otherwise(0)

).withColumn(

    "long_encounter_duration_flag",

    when(col("avg_encounter_duration_hours") >= 24, 1).otherwise(0)
)

display(encounter_utilization_df)

patient_id,total_encounters,ambulatory_encounters,emergency_encounters,inpatient_encounters,avg_encounter_duration_hours,max_encounter_duration_hours,total_encounter_hours,high_emergency_utilization_flag,high_inpatient_utilization_flag,high_total_utilization_flag,long_encounter_duration_flag
b0a06ead-cc42-aa48-dad6-841d4aa679fa,32,30,2,0,0.30257812500000003,1.0,9.682500000000001,0,0,0,0
ccfc4db2-2026-7adb-3db0-33f3828140bb,38,37,1,0,0.26973684210526316,1.0,10.25,0,0,0,0
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,64,63,0,1,0.62109375,24.0,39.75,0,0,1,0
71a8b156-760b-df6b-859e-eefc7932a526,19,18,1,0,0.2894736842105263,1.0,5.5,0,0,0,0
76b289fd-e825-734c-8446-316f59643593,24,21,2,1,9.409189814814814,216.56722222222223,225.82055555555556,0,0,0,0
92fb7efc-5cfd-f8d3-927b-42f8ee099531,27,25,2,0,0.35032921810699585,1.0947222222222222,9.458888888888888,0,0,0,0
81aa7647-779f-fd6b-94cf-782e606efeb2,17,17,0,0,0.25,0.25,4.25,0,0,0,0
346a1435-2455-914f-c287-7b88052d05db,69,65,4,0,0.3047342995169082,1.0,21.026666666666667,1,0,1,0
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,38,37,1,0,5.659824561403509,205.0,215.07333333333335,0,0,0,0
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,54,48,6,0,123.94552469135805,6672.0,6693.058333333334,1,0,1,1


## Step 6 — Validate Encounter Utilization Metrics

### What We Are Doing
We are validating enterprise healthcare utilization KPIs.

### Why We Are Doing This
We need to confirm realistic operational healthcare metrics.

### Expected Output
Healthcare utilization KPI summary.

In [0]:
display(

    encounter_utilization_df.select(

        avg("total_encounters").alias("avg_total_encounters"),

        avg("ambulatory_encounters").alias("avg_ambulatory_encounters"),

        avg("emergency_encounters").alias("avg_emergency_encounters"),

        avg("inpatient_encounters").alias("avg_inpatient_encounters"),

        avg("avg_encounter_duration_hours").alias("avg_encounter_duration_hours"),

        avg("total_encounter_hours").alias("avg_total_encounter_hours")
    )

)

avg_total_encounters,avg_ambulatory_encounters,avg_emergency_encounters,avg_inpatient_encounters,avg_encounter_duration_hours,avg_total_encounter_hours
50.11171171171171,47.009009009009006,1.8468468468468469,1.255855855855856,1.9318904835095931,131.060898898899


## Step 7 — Analyze Encounter Class Distribution

### What We Are Doing
We are analyzing encounter utilization distribution.

### Why We Are Doing This
This supports:
- operational analytics
- healthcare planning
- utilization management

### Expected Output
Encounter class distribution summary.

In [0]:
display(

    encounter_df.groupBy(
        "encounter_class"
    ).count().orderBy(
        desc("count")
    )

)

encounter_class,count
AMB,26090
EMER,1025
IMP,697


## Step 8 — Save Gold Encounter Utilization Table

### What We Are Doing
We are saving the Gold utilization analytics table.

### Why We Are Doing This
This table becomes:
- Power BI source
- SQL analytics source
- ML utilization feature store
- operational healthcare dashboard source

### Expected Output
A Delta table:

`healthcare_catalog.gold.encounter_utilization_summary`

In [0]:
encounter_utilization_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.encounter_utilization_summary"
    )

print("Gold encounter_utilization_summary saved successfully.")

Gold encounter_utilization_summary saved successfully.


## Step 9 — Verify Gold Tables

### What We Are Doing
We are verifying Gold tables.

### Why We Are Doing This
We want to confirm successful Gold table creation.

### Expected Output
`encounter_utilization_summary` should appear in Gold schema.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.gold
""").show(truncate=False)

+--------+-----------------------------+-----------+
|database|tableName                    |isTemporary|
+--------+-----------------------------+-----------+
|gold    |encounter_utilization_summary|false      |
|gold    |patient_summary              |false      |
+--------+-----------------------------+-----------+

